# Validação 05 — Resumo e texto completo no PMC

## Goal

Comprovar que um artigo do PubMed pode ter seu resumo recuperado e, quando existir PMCID, seu texto completo obtido e organizado em seções pelo PubMed Central.

## Setup

Fontes: [PMC ID Converter](https://pmc.ncbi.nlm.nih.gov/tools/id-converter-api/) e [EFetch oficial do NCBI](https://www.ncbi.nlm.nih.gov/books/NBK25499/#chapter4.EFetch). O notebook usa o PMID `33431520`, que possui o registro `PMC7805365`. Apenas métricas e trechos curtos são exibidos.

In [1]:
from datetime import datetime, timezone
from pathlib import Path
from pprint import pprint
import os
import sys

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root / "src"))

from fatofake import (
    PmcClient,
    PubMedClient,
    prepare_search_plan,
    retrieve_article_content,
    search_pubmed,
    validate_analysis_input,
)

executed_at = datetime.now(timezone.utc).isoformat()
print(f"Execução UTC: {executed_at}")

Execução UTC: 2026-09-23T13:40:27.315315+00:00


## Steps

Recuperamos o artigo pelo PMID, buscamos seu resumo, convertemos PMID para PMCID e solicitamos o XML completo ao EFetch. O conteúdo é marcado como `FULL_TEXT` ou `ABSTRACT_ONLY`.

In [2]:
class PmidQueryPlanner:
    def generate_queries(self, claim: str) -> list[str]:
        return ["33431520[pmid]"]

In [3]:
analysis_input = validate_analysis_input(
    "O consumo de café altera o risco de câncer de próstata."
)
search_plan = prepare_search_plan(analysis_input, PmidQueryPlanner())
pubmed_result = search_pubmed(
    search_plan,
    PubMedClient(
        email=os.getenv("NCBI_EMAIL"),
        api_key=os.getenv("NCBI_API_KEY"),
    ),
    max_results_per_query=1,
)
publication = pubmed_result.publications[0]

In [4]:
content = retrieve_article_content(
    publication,
    PmcClient(
        email=os.getenv("NCBI_EMAIL"),
        api_key=os.getenv("NCBI_API_KEY"),
    ),
)

content_summary = {
    "pmid": content.pmid,
    "pmcid": content.pmcid,
    "doi": content.doi,
    "access_level": content.access_level,
    "abstract_characters": len(content.abstract or ""),
    "full_text_characters": len(content.full_text or ""),
    "section_count": len(content.sections),
    "section_titles": [section.title for section in content.sections],
    "pubmed_url": content.pubmed_url,
    "pmc_url": content.pmc_url,
}
pprint(content_summary)

{'abstract_characters': 1894,
 'access_level': 'FULL_TEXT',
 'doi': '10.1136/bmjopen-2020-038902',
 'full_text_characters': 24708,
 'pmc_url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC7805365/',
 'pmcid': 'PMC7805365',
 'pmid': '33431520',
 'pubmed_url': 'https://pubmed.ncbi.nlm.nih.gov/33431520/',
 'section_count': 5,
 'section_titles': ['Introduction',
                    'Methods',
                    'Results',
                    'Discussion',
                    'Conclusions']}


In [5]:
print("Resumo — início:")
print((content.abstract or "indisponível")[:500])

print("\nPrimeira seção — início:")
if content.sections:
    print(f"{content.sections[0].title}: {content.sections[0].text[:500]}")
else:
    print("indisponível")

Resumo — início:
OBJECTIVES: To conduct a systematic review with meta-analysis of cohort studies to evaluate the association of coffee consumption with the risk of prostate cancer.

DATA SOURCES: PubMed, Web of Science and Embase were searched for eligible studies up to September 2020.

STUDY SELECTION: Cohort studies were included.

DATA EXTRACTION AND SYNTHESIS: Two researchers independently reviewed the studies and extracted the data. Data synthesis was performed via systematic review and meta-analysis of eli

Primeira seção — início:
Introduction: Prostate cancer is the second most frequently diagnosed cancer and the sixth leading cause of cancer death in men. There were 1 276 000 new cancer cases and 359 000 cancer deaths in 2018.1 It is estimated that nearly three-quarters of prostate cancer cases occur in developed countries.1 Since the 1970s, the incidence of prostate cancer has also increased rapidly in some Asian countries such as China, Singapore and Japan, where the inciden

## Checks

As verificações confirmam o relacionamento PMID→PMCID, a presença do resumo, do corpo completo e de seções utilizáveis nas próximas etapas.

In [6]:
assert content.pmid == "33431520"
assert content.pmcid == "PMC7805365"
assert content.access_level == "FULL_TEXT"
assert content.abstract and len(content.abstract) > 100
assert content.full_text and len(content.full_text) > 1000
assert content.sections
assert content.pmc_url == "https://pmc.ncbi.nlm.nih.gov/articles/PMC7805365/"

print(
    f"Conteúdo recuperado: {len(content.abstract)} caracteres de resumo e "
    f"{len(content.full_text)} caracteres de texto completo."
)

Conteúdo recuperado: 1894 caracteres de resumo e 24708 caracteres de texto completo.


## Next Steps

A recuperação de conteúdo estará validada quando todas as células forem executadas sem erros. A próxima etapa será dividir o texto em trechos rastreáveis para busca e análise de evidências.